# 🎯 [K12_M10] 행정 업무 FAQ 챗봇 완성하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m10_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 10. 행정 업무 FAQ 챗봇 완성하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 10. 행정 업무 FAQ 챗봇 완성하기
**소요시간**: 약 1.5시간

![미션 10 부대 행정 업무 FAQ 챗봇 4대 컴포넌트 표준 설계서 및 인간 책임 원칙(HITL) 다이어그램](/uploads/common/K12_m10_chatbot_architecture.png)

### 📋 미션 개요 및 배경
그동안 단계별로 구축하고 검증한 행정 FAQ 데이터, 시스템 프롬프트, 예외 안내 규칙, 테스트 결과서, 갱신 관리 지침을 집대성하여 최종 '행정 업무 FAQ 챗봇 설계서'를 완성합니다. 'AI 답변은 행정 참고용이며 최종 결정은 담당 부서 및 지휘계통을 따른다'는 인간 책임 원칙(HITL) 고지문을 포함합니다. 부대 행정 현장에서 즉각 배포 및 실무 적용이 가능한 완결된 AI 행정 솔루션 기획·설계 산출물을 완성합니다.

### 🎯 미션 학습목표
FAQ 표, 시스템 프롬프트(역할·근거·형식·예외 규칙), 시험 결과, 갱신 규칙을 하나의 설계서로 묶습니다. '챗봇 답변은 참고용이며 최종 확인은 담당 부서·지휘계통'이라는 안내 문구를 넣어 완성합니다. **이를 통해 단위 기능들을 하나의 완전한 표준 설계서로 통합하고, 인간 책임 원칙(HITL) 고지문을 포함하여 부대 실무에 즉시 적용 가능한 AI 시스템 패키징 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. FAQ·프롬프트·예외 규칙·형식을 설계서로 묶는다.
#### 이 단계의 역할

앞서 구축한 행정 FAQ 표준 데이터, 시스템 프롬프트 전문, 예외 Fallback 규칙, 단계별 출력 템플릿을 하나의 완결된 '행정 업무 FAQ 챗봇 설계서'로 통합합니다.

#### 해야 할 일

- 설계서의 표준 목차(개요, FAQ 데이터셋 구조, 시스템 프롬프트 전문, 출력 서식, 예외 처리 규칙)를 구성합니다.

- 미션 1부터 6까지 완성한 산출물들을 목차에 맞게 체계적으로 배치합니다.

- AI에게 전체 설계서의 문서 구조와 서식이 일관되고 전문적인지 검토받습니다.

#### 💡 AI 프롬프트 질의 예시

> **[행정 FAQ 챗봇 통합 설계서 편찬]:** "지금까지 완성한 1) FAQ 표, 2) 시스템 프롬프트 전문, 3) Fallback 규칙, 4) Few-shot 출력 서식을 하나의 체계적인 '부대 행정 FAQ 챗봇 시스템 설계서' 마크다운 문서로 통합 구성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **산출물 누락 점검:** FAQ 20건, 시스템 프롬프트 지시문, 출력 템플릿이 빠짐없이 수록되었는가?
- **구조적 완성도:** 다른 실무자가 설계서만 보고도 동일한 챗봇을 즉각 복제·운용할 수 있는 수준인가?

#### 핵심 확인 지표

> 행정 업무 FAQ 챗봇 표준 설계서 초안

#### 학습자가 제출해야 하는 내용

- 통합 설계서 초안

#### 🔹 Step 2. 시험 결과와 갱신 규칙을 넣는다.
#### 이 단계의 역할

12개 테스트 결과 보고서(정확도 지표 및 취약점 분석)와 행정 규정 개정 시 데이터 유지보수 및 갱신 프로세스를 설계서 부록으로 첨부합니다.

#### 해야 할 일

- 미션7에서 산출한 12개 시험 질문 정량 평가 결과(정확도 %)와 주요 튜닝 내역을 요약합니다.

- 미션8에서 수립한 FAQ 개정 일자 표기, 변경 이력 로그, 버전 관리 규칙을 유지보수 섹션에 배치합니다.

- AI에게 품질 보증 및 유지보수 절차가 명확히 서술되었는지 확인을 요청합니다.

#### 💡 AI 프롬프트 질의 예시

> **[검증 결과 및 유지보수 지침 문서화]:** "챗봇의 신뢰도를 입증하는 12개 벤치마크 테스트 결과(정확도, 오답 개선 내역)와 분기별 FAQ 데이터 갱신 및 형상 관리 지침을 설계서의 운영 관리 섹션으로 작성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정량 지표 제시:** 테스트 정확도 수치와 주요 개선 사례가 객관적 데이터로 제시되었는가?
- **유지보수 실효성:** 담당자 변경 시에도 지속적으로 데이터를 갱신할 수 있는 절차가 담겼는가?

#### 핵심 확인 지표

> 품질 검증 보고 및 유지보수 지침이 포함된 설계서

#### 학습자가 제출해야 하는 내용

- 품질 및 운영 지침이 보강된 설계서

#### 🔹 Step 3. 확인 안내 문구를 넣는다.
#### 이 단계의 역할

AI 환각 및 규정 변경에 따른 군사 행정 혼선을 원천 방지하기 위해, 모든 답변 및 문서 말미에 인간 책임 원칙(HITL) 안내 고지문을 필수로 삽입합니다.

#### 해야 할 일

- "본 챗봇의 답변은 참고용이며, 최종 절차 및 승인은 해당 담당 부서와 지휘계통을 따른다"는 면책 및 확인 고지문을 작성합니다.

- 설계서의 주의사항 섹션과 시스템 프롬프트의 기본 출력 규칙 맨 마지막에 해당 고지문을 영구 삽입합니다.

- 군사보안 준수 수칙(실제 개인정보 입력 금지)을 재확인합니다.

#### 💡 AI 프롬프트 질의 예시

> **[인간 책임 원칙(HITL) 고지문 설계]:** "챗봇의 모든 안내 하단과 시스템 설계서에 필수 명시할 고지문을 작성해 줘. '본 챗봇의 답변은 신속한 행정 절차 확인을 돕는 보조 자료이며, 최종 행정 효력과 규정 승인은 담당 부서 및 지휘계통을 따른다'는 원칙을 정중하고 엄격하게 표현해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **HITL 원칙 준수:** AI 결과의 보조성과 최종 인간 책임 원칙이 타협 없이 명문화되었는가?
- **보안 경고 배치:** 실제 부대 비밀이나 개인정보 입력 금지 조항이 명시되었는가?

#### 핵심 확인 지표

> 인간 책임 원칙(HITL) 고지문이 완비된 최종 완성 설계서

#### 학습자가 제출해야 하는 내용

- 행정 업무 FAQ 챗봇 최종 설계서 완성본

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

차기 부대 관리자나 후임 행정병이 인수인계받아 즉시 운용할 수 있도록 1페이지 요약 치트시트(Cheat-sheet)를 제작합니다.

#### 해야 할 일

- 전체 설계서의 핵심 프롬프트와 운영 수칙을 A4 1장 분량의 핵심 요약본으로 축약합니다.

- 행정반 데스크에 부착할 수 있는 깔끔한 운용 가이드를 완성합니다.

#### 💡 AI 프롬프트 질의 예시

> **[1페이지 운영 요약 치트시트 생성]:** "후임 행정병이 즉시 보고 챗봇을 관리할 수 있도록 시스템 프롬프트 핵심 규칙과 FAQ 갱신법을 A4 1페이지 분량의 요약 치트시트로 압축해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **인수인계 용이성:** 핵심 프롬프트와 주의사항이 한눈에 파악되는가?

#### 핵심 확인 지표

> 1페이지 챗봇 운영 치트시트

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **최종 '행정 업무 FAQ 챗봇 표준 설계서' 종합 서식 (MD):** FAQ 데이터셋, 최종 시스템 프롬프트, Fallback 규칙, 출력 템플릿 통합 패키지 서식
- **챗봇 시스템 성능 검증 보고서 요약 카드 양식:** 12개 벤치마크 테스트 결과 지표 및 프롬프트 튜닝 이력 요약표
- **[학습 가이드] 책임 있는 AI(Responsible AI)와 행정 챗봇 배포 가이드:** 인간 책임 원칙(HITL) 고지문, 군사 보안 수칙 준수 서약, 후임 행정병 인수인계 운용 매뉴얼

**실습 팁:** 설계서 맨 앞과 챗봇의 모든 응답 하단에는 반드시 "본 챗봇의 답변은 참고용이며, 최종 행정 승인 및 절차는 해당 부서와 지휘계통을 따른다"는 HITL 고지문을 명시하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 결과물 완성 — 챗봇 핵심 구성 요소를 완결된 시스템 설계서로 통합하고, 인간 책임 원칙(HITL) 고지문을 필수 반영했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 행정 업무 FAQ 챗봇 최종 표준 설계서 완성본 도출

- 데이터, 프롬프트, 서식, 예외규칙, 검증보고서의 완전한 통합

- 인간 책임 원칙(HITL) 및 군사보안 준수 안내문 영구 탑재

#### 채점자 판정 메모

> 설계서 및 프롬프트에 "본 챗봇 답변은 참고용이며 최종 승인은 지휘계통을 따른다"는 HITL 고지문이 빠짐없이 명시되었는지 점검한다.



In [ ]:
# [미션 10.] 학생 실습 - 프롬프트 및 결과 기록 셀
# 💡 [데이터셋 확인 팁]: 아래 주석을 해제하여 본 미션의 가상 데이터셋 요약을 확인할 수 있습니다.
# show_dataset_summary(10)

# TODO 1: 위 실습 가이드와 AI 질의 예시를 참고하여 작성한 프롬프트를 입력하세요.
mission_10_prompt = """
# 여기에 본인이 설계한 프롬프트를 작성하세요.
"""

# TODO 2: 생성형 AI가 출력한 답변 및 원문 대조(Fact-check) 검증 내역을 기록하세요.
mission_10_result = """
# 여기에 AI 생성 결과 및 팩트체크 대조표/산출물을 마크다운 서식으로 기록하세요.
"""

# 렌더러 실행 (작성한 내용을 시각적으로 확인합니다)
show_result("미션 10. 행정 업무 FAQ 챗봇 완성하기", mission_10_prompt, mission_10_result)

